# CHESCA vs Reserve-Contract Peer Mesh

이 notebook은 기존 결과를 덮어쓰지 않는 새 실행 파일입니다. 다운로드한 공식 `CHESCA-main`, 기존 unrestricted mesh, 그리고 CHESCA의 시간대별 최소 SOC를 peer 계약으로 보존하는 새 `reserve-contract mesh`를 비교합니다.

## 1. Google Drive 연결

`chesca_vs_mesh` 폴더 전체를 `MyDrive` 바로 아래에 올린 뒤 이 notebook을 실행합니다.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/chesca_vs_mesh')
OFFICIAL_DIR = PROJECT_DIR / 'CHESCA-main'
assert (OFFICIAL_DIR / 'checa' / 'agent.py').exists(), f'공식 CHESCA 폴더를 찾을 수 없습니다: {OFFICIAL_DIR}'
assert (PROJECT_DIR / 'src' / 'chesca_vs_mesh' / 'reserve_mesh_agent.py').exists(), 'reserve-contract 코드가 누락되었습니다.'
print('Project:', PROJECT_DIR)
print('Official source:', OFFICIAL_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/chesca_vs_mesh
Official source: /content/drive/MyDrive/chesca_vs_mesh/CHESCA-main


## 2. Colab 설치

CityLearn 2.1b12 런타임은 프로젝트의 `third_party`에 포함되어 있습니다. 오래된 PyPI 의존성이 Colab의 NumPy/Pandas 바이너리와 충돌하지 않도록 CityLearn 자체는 pip로 설치하지 않습니다.

In [2]:
%pip install -q "gym==0.26.2" "simplejson>=3.19" "xgboost>=1.7,<3"

import sys
VENDORED_CITYLEARN = PROJECT_DIR / 'third_party' / 'CityLearn-2.1b12'
assert (VENDORED_CITYLEARN / 'citylearn' / 'citylearn.py').exists(), f'CityLearn runtime을 찾을 수 없습니다: {VENDORED_CITYLEARN}'
sys.path.insert(0, str(VENDORED_CITYLEARN))

import numpy as np
import pandas as pd
import scipy
import torch
import xgboost
import citylearn
from citylearn.citylearn import CityLearnEnv

print('numpy:', np.__version__, 'pandas:', pd.__version__, 'scipy:', scipy.__version__)
print('torch:', torch.__version__, 'xgboost:', xgboost.__version__)
print('CityLearn:', citylearn.__version__, citylearn.__file__)
assert citylearn.__version__ == '2.1b12'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 18.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


numpy: 2.0.2 pandas: 2.2.2 scipy: 1.16.3
torch: 2.11.0+cu128 xgboost: 2.1.4
CityLearn: 2.1b12 /content/drive/MyDrive/chesca_vs_mesh/third_party/CityLearn-2.1b12/citylearn/__init__.py


## 3. 실험 설정

`chesca_reserve_contract_mesh`는 원본 CHESCA가 정한 `min_soc_per_hour[(hour + 1) % 24]` 아래로 내려가는 추가 방전 메시지를 생성하지 않습니다. 새 결과는 별도 태그에 저장되므로 기존 `official_chesca_vs_peer_mesh` 결과는 유지됩니다.

In [3]:
import sys
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from chesca_vs_mesh import MeshConfig, available_datasets
from chesca_vs_mesh.reserve_mesh_agent import ReserveContractConfig
from chesca_vs_mesh.reserve_evaluation import ReserveBenchmarkSuite

print('Available bundled datasets:')
print(available_datasets())

DATASET = 'citylearn_challenge_2023_phase_3_1'
EPISODE_STEPS = None  # None: official schema 전체 기간. 빠른 연결 확인은 71 등으로 변경.
TAG = 'reserve_contract_comparison_v1'
CONTROLLERS = ['chesca_official', 'chesca_mesh', 'chesca_reserve_contract_mesh']

previous_mesh_config = MeshConfig(
    rounds=3,
    offer_step=0.04,
    target_quantile=0.65,
    peak_weight=1.00,
    ramp_weight=0.32,
    price_weight=0.10,
    carbon_weight=0.08,
)
reserve_config = ReserveContractConfig(
    rounds=3,
    offer_step=0.04,
    target_quantile=0.65,
    peak_weight=1.00,
    ramp_weight=0.32,
    price_weight=0.10,
    carbon_weight=0.08,
)
suite = ReserveBenchmarkSuite(
    output_directory=PROJECT_DIR / 'results',
    mesh_config=previous_mesh_config,
    reserve_config=reserve_config,
    power_outage_seed=None,
)

Available bundled datasets:
['citylearn_challenge_2023_phase_1', 'citylearn_challenge_2023_phase_2_local_evaluation', 'citylearn_challenge_2023_phase_2_online_evaluation_1', 'citylearn_challenge_2023_phase_2_online_evaluation_2', 'citylearn_challenge_2023_phase_2_online_evaluation_3', 'citylearn_challenge_2023_phase_3_1', 'citylearn_challenge_2023_phase_3_2', 'citylearn_challenge_2023_phase_3_3', 'warm_up']


## 4. 단일 schema 비교

`challenge_cost`와 모든 `*_change_vs_chesca_pct`는 낮을수록 좋습니다. 새 mesh가 기존 mesh의 grid 개선을 유지하면서 `resilience_cost` 악화를 줄이는지 먼저 확인합니다.

In [4]:
result = suite.compare_controllers(
    dataset_name=DATASET,
    controllers=CONTROLLERS,
    episode_steps=EPISODE_STEPS,
    tag=TAG,
)

score_columns = [
    'controller', 'challenge_cost', 'comfort_cost', 'emissions_cost',
    'grid_cost', 'resilience_cost', 'challenge_cost_change_vs_chesca_pct',
    'grid_cost_change_vs_chesca_pct', 'resilience_cost_change_vs_chesca_pct',
]
display(result.summary[score_columns])
display(result.citylearn_metrics)
print('Results saved to:', result.output_directory)

,controller,challenge_cost,comfort_cost,emissions_cost,grid_cost,resilience_cost,challenge_cost_change_vs_chesca_pct,grid_cost_change_vs_chesca_pct,resilience_cost_change_vs_chesca_pct
0,chesca_official,0.590169,0.130315,0.925583,0.957768,0.570619,0.000000,0.000000,0.000000
1,chesca_mesh,0.589098,0.130710,0.925827,0.943553,0.580788,-0.181399,-1.484100,1.782102
2,chesca_reserve_contract_mesh,0.594425,0.130815,0.925844,0.957076,0.584910,0.721172,-0.072254,2.504565


,controller,carbon_emissions_total,discomfort_proportion,ramping_average,daily_one_minus_load_factor_average,daily_peak_average,annual_peak_average,one_minus_thermal_resilience_proportion,power_outage_normalized_unserved_energy_total,average_score
0,chesca_official,0.925583,0.130315,0.850411,0.958645,0.889024,1.132991,0.788781,0.352457,0.590169
1,chesca_mesh,0.925827,0.130710,0.819864,0.960908,0.891153,1.102288,0.805797,0.355779,0.589098
2,chesca_reserve_contract_mesh,0.925844,0.130815,0.832835,0.964752,0.895835,1.134880,0.814130,0.355691,0.594425


Results saved to: /content/drive/MyDrive/chesca_vs_mesh/results/citylearn_challenge_2023_phase_3_1/reserve_contract_comparison_v1


## 5. Reserve 계약이 실제로 작동했는지 확인

`reserve_limited_peers`는 해당 step에서 공식 reserve 때문에 추가 방전 제안을 낼 수 없었던 peer 수입니다. `selected_reserve_margin_min`이 음수이면 새 협상이 원본 reserve를 훼손했는지 확인해야 합니다. 단, 원본 CHESCA 자체가 이미 reserve 아래인 경우에는 `official_reserve_margin_min`도 함께 음수가 됩니다.

In [5]:
reserve_negotiations = result.negotiations[
    result.negotiations['controller'] == 'chesca_reserve_contract_mesh'
].copy()
if reserve_negotiations.empty:
    print('Reserve-contract negotiation log가 없습니다.')
else:
    diagnostic_columns = [
        'changed_peers', 'relief_selected_peers', 'reserve_limited_peers',
        'protected_reserve_soc', 'official_reserve_margin_min',
        'selected_reserve_margin_min', 'predicted_grid_delta',
    ]
    display(reserve_negotiations[diagnostic_columns].describe())
    display(reserve_negotiations.tail(20))

reserve_messages = result.messages[
    result.messages['controller'] == 'chesca_reserve_contract_mesh'
]
if not reserve_messages.empty:
    display(reserve_messages.tail(20))

,changed_peers,relief_selected_peers,reserve_limited_peers,protected_reserve_soc,official_reserve_margin_min,selected_reserve_margin_min,predicted_grid_delta
count,2163.000000,2163.000000,2163.000000,2163.000000,2163.000000,2163.000000,2163.000000
mean,2.516875,1.495608,2.431345,0.669154,0.078675,0.083122,-0.068100
std,2.498671,2.288077,2.459718,0.084536,0.123286,0.122389,0.511799
min,0.000000,0.000000,0.000000,0.500000,-0.092441,-0.092441,-0.876000
25%,0.000000,0.000000,0.000000,0.600000,0.000000,0.000000,-0.424000
50%,2.000000,0.000000,1.000000,0.650000,0.000000,0.040000,0.000000
75%,5.000000,3.000000,5.000000,0.720000,0.144000,0.135000,0.132000
max,6.000000,6.000000,6.000000,0.850000,0.490000,0.490000,0.876000


,controller,step,hour,active_peers,changed_peers,official_predicted_grid,negotiated_predicted_grid,predicted_grid_delta,district_target,final_shadow_signal,logical_message_count,relief_selected_peers,reserve_limited_peers,protected_reserve_soc,official_reserve_margin_min,selected_reserve_margin_min,selected_reserve_margin_mean
4306,chesca_reserve_contract_mesh,2187,4,6,6,7.577499,8.453499,8.760000e-01,9.558321,-0.112484,90,0.0,4.0,0.85,0.000000,0.040000,0.063333
4307,chesca_reserve_contract_mesh,2188,5,6,4,5.847704,6.431704,5.840000e-01,9.558321,-0.144926,90,0.0,0.0,0.80,0.095000,0.135000,0.169167
4308,chesca_reserve_contract_mesh,2189,6,6,0,6.203592,6.203592,0.000000e+00,9.558321,0.029503,90,0.0,0.0,0.75,0.192000,0.192000,0.224000
4309,chesca_reserve_contract_mesh,2190,7,6,1,5.572979,5.732979,1.600000e-01,9.558321,-0.097790,90,0.0,0.0,0.70,0.241667,0.281667,0.288611
4310,chesca_reserve_contract_mesh,2191,8,6,0,3.912347,3.912347,4.440892e-16,9.558321,-0.054973,90,0.0,0.0,0.60,0.390000,0.390000,0.390000
4311,chesca_reserve_contract_mesh,2192,9,6,0,3.467042,3.467042,0.000000e+00,9.558321,0.012158,90,0.0,0.0,0.50,0.490000,0.490000,0.490000
4312,chesca_reserve_contract_mesh,2193,10,6,0,3.656058,3.656058,-4.440892e-16,9.558321,-0.073783,90,0.0,0.0,0.60,0.390000,0.390000,0.390000
4313,chesca_reserve_contract_mesh,2194,11,6,0,3.592305,3.592305,0.000000e+00,9.558321,0.061279,90,0.0,0.0,0.65,0.291429,0.291429,0.331905
4314,chesca_reserve_contract_mesh,2195,12,6,0,5.263051,5.263051,0.000000e+00,9.558321,0.075979,90,0.0,0.0,0.65,0.194286,0.194286,0.315714
4315,chesca_reserve_contract_mesh,2196,13,6,6,5.933145,5.123145,-8.100000e-01,9.558321,0.149308,90,6.0,0.0,0.70,0.048333,0.008333,0.196944


,controller,step,round_id,sender,official_grid,proposed_grid,lower_grid,upper_grid,soc,district_proposal,district_target,shadow_signal,recipient_count,protected_reserve_soc,official_reserve_margin,offered_discharge_flex_soc
77848,chesca_reserve_contract_mesh,2205,2,4,3.011562,3.011562,3.011562,3.171562,0.599935,9.665339,9.553435,-0.377394,5,0.55,0.0,0.0
77849,chesca_reserve_contract_mesh,2205,2,5,1.598442,1.598442,1.598442,1.730442,0.598984,9.665339,9.553435,-0.377394,5,0.55,0.0,0.0
77850,chesca_reserve_contract_mesh,2206,0,0,1.541803,1.541803,1.541803,1.701803,0.589284,10.164423,9.558321,-0.033135,5,0.60,0.0,0.0
77851,chesca_reserve_contract_mesh,2206,0,1,0.804086,0.804086,0.804086,0.964086,0.636417,10.164423,9.558321,-0.033135,5,0.60,0.0,0.0
77852,chesca_reserve_contract_mesh,2206,0,2,1.133793,1.133793,1.133793,1.265793,0.588504,10.164423,9.558321,-0.033135,5,0.60,0.0,0.0
77853,chesca_reserve_contract_mesh,2206,0,3,2.075204,2.075204,2.075204,2.207204,0.588504,10.164423,9.558321,-0.033135,5,0.60,0.0,0.0
77854,chesca_reserve_contract_mesh,2206,0,4,2.828438,2.828438,2.828438,2.988438,0.589284,10.164423,9.558321,-0.033135,5,0.60,0.0,0.0
77855,chesca_reserve_contract_mesh,2206,0,5,1.781099,1.781099,1.781099,1.913099,0.588504,10.164423,9.558321,-0.033135,5,0.60,0.0,0.0
77856,chesca_reserve_contract_mesh,2206,1,0,1.541803,1.541803,1.541803,1.701803,0.589284,10.164423,9.558321,-0.033135,5,0.60,0.0,0.0
77857,chesca_reserve_contract_mesh,2206,1,1,0.804086,0.804086,0.804086,0.964086,0.636417,10.164423,9.558321,-0.033135,5,0.60,0.0,0.0


## 6. 논문형 Public Cost / Private Cost 비교

Public 및 Private 각각 3개 schema에 세 controller를 모두 실행하므로 시간이 오래 걸립니다. 이미 확인한 private schema를 사용한 개선 결과는 개발 중 검증 결과로 해석하고, 새로운 unseen test를 대신한다고 주장하지 않습니다.

In [6]:
RUN_PUBLIC_PRIVATE = True

if RUN_PUBLIC_PRIVATE:
    leaderboard = suite.compare_public_private_costs(
        controllers=CONTROLLERS,
        episode_steps=None,
        tag='paper_public_private_reserve_contract_v1',
    )
    display(leaderboard.paper_table)
    display(leaderboard.summary)

    diagnostic_columns = [
        'split', 'run_id', 'dataset', 'controller', 'challenge_cost',
        'grid_cost', 'resilience_cost',
        'grid_cost_change_vs_chesca_pct',
        'resilience_cost_change_vs_chesca_pct',
    ]
    display(leaderboard.runs[diagnostic_columns])
    print('Public/private results saved to:', leaderboard.output_directory)
else:
    print('RUN_PUBLIC_PRIVATE=True로 변경하면 Public/Private Cost 비교를 실행합니다.')

,controller,Private Cost,Public Cost,Private Cost Change vs CHESCA (%),Public Cost Change vs CHESCA (%)
0,chesca_mesh,0.566371,0.502961,-0.239082,-1.071209
1,chesca_official,0.567729,0.508408,0.000000,0.000000
2,chesca_reserve_contract_mesh,0.570473,0.504509,0.483396,-0.766910


,split,controller,leaderboard_cost,comfort_cost,emissions_cost,grid_cost,resilience_cost,leaderboard_cost_change_vs_chesca_pct
0,private,chesca_mesh,0.566371,0.129195,0.928669,0.920573,0.528580,-0.239082
1,private,chesca_official,0.567729,0.128942,0.928365,0.934023,0.520009,0.000000
2,private,chesca_reserve_contract_mesh,0.570473,0.129205,0.928820,0.934217,0.528549,0.483396
3,public,chesca_mesh,0.502961,0.070449,0.951868,0.879749,0.409050,-1.071209
4,public,chesca_official,0.508408,0.070520,0.951344,0.888128,0.418929,0.000000
5,public,chesca_reserve_contract_mesh,0.504509,0.070398,0.952034,0.885829,0.408123,-0.766910


,split,run_id,dataset,controller,challenge_cost,grid_cost,resilience_cost,grid_cost_change_vs_chesca_pct,resilience_cost_change_vs_chesca_pct
0,public,1,citylearn_challenge_2023_phase_2_online_evalua...,chesca_official,0.536594,0.885566,0.513373,0.000000,0.000000
1,public,1,citylearn_challenge_2023_phase_2_online_evalua...,chesca_mesh,0.533249,0.874186,0.513388,-1.285026,0.002899
2,public,1,citylearn_challenge_2023_phase_2_online_evalua...,chesca_reserve_contract_mesh,0.534136,0.877143,0.513371,-0.951113,-0.000435
3,public,2,citylearn_challenge_2023_phase_2_online_evalua...,chesca_official,0.520862,0.875704,0.471915,0.000000,0.000000
4,public,2,citylearn_challenge_2023_phase_2_online_evalua...,chesca_mesh,0.518596,0.868274,0.471619,-0.848515,-0.062703
5,public,2,citylearn_challenge_2023_phase_2_online_evalua...,chesca_reserve_contract_mesh,0.521617,0.877717,0.472225,0.229793,0.065552
6,public,3,citylearn_challenge_2023_phase_2_online_evalua...,chesca_official,0.467766,0.903115,0.271499,0.000000,0.000000
7,public,3,citylearn_challenge_2023_phase_2_online_evalua...,chesca_mesh,0.457040,0.896788,0.242143,-0.700484,-10.812737
8,public,3,citylearn_challenge_2023_phase_2_online_evalua...,chesca_reserve_contract_mesh,0.457772,0.902628,0.238773,-0.053934,-12.053812
9,private,1,citylearn_challenge_2023_phase_3_1,chesca_official,0.590169,0.957768,0.570619,0.000000,0.000000


Public/private results saved to: /content/drive/MyDrive/chesca_vs_mesh/results/paper_public_private_reserve_contract_v1
